<a href="https://colab.research.google.com/github/Amyerm/ClassFiles/blob/main/Sesion12_Evaluacion_Datos_Categoricos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Programación para Analítica Descriptiva y Predictiva**
**Maestría en Inteligencia Artificial y Analítica de Datos**

# Sesión 12: Evaluación — Limpieza y Transformación de Datos Categóricos

**Entrega individual**

- **Nombre**: Ari Daniel Mendoza Enrriquez
- **Matrícula** 271756

Esta evaluación aplica los tres temas de la Sesión 12 (errores tipográficos y valores inconsistentes, alta cardinalidad, tipos incorrectos) a un dataset que no se trabajó en clase: **Telco Customer Churn**.


No hay una única respuesta correcta en varias de las actividades — lo que se evalúa es que la conclusión esté respaldada por el código que la sustenta, no solo la conclusión en sí.

## Preparación

In [ ]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


---
## Actividad 1 — Formato y valores inconsistentes (15 pts)

Revisa **todas** las columnas categóricas del dataset (no elijas solo una) en busca de variantes de formato (mayúsculas, espacios) que deberían normalizarse.

In [ ]:
# Seleccionar las columnas categóricas de texto
columnas_categoricas = df.select_dtypes(include='object').columns.tolist()

# Estas no se consideran categoricas
columnas_categoricas.remove('customerID')
columnas_categoricas.remove('TotalCharges')

# Es categórica aunque esté representada con 0 y 1
columnas_categoricas.append('SeniorCitizen')

# Revisar valores de cada columna categórica
for columna in columnas_categoricas:
    print("\nColumna:", columna)
    print(df[columna].value_counts(dropna=False))


Columna: gender
gender
Male      3555
Female    3488
Name: count, dtype: int64

Columna: Partner
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Columna: Dependents
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

Columna: PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

Columna: MultipleLines
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

Columna: InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

Columna: OnlineSecurity
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

Columna: OnlineBackup
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

Columna: DeviceProtection
DeviceProtection
No                     3095
Yes                    2422
No internet s

**1.2 — Conclusión (responde aquí en Markdown):**

¿Encontraste alguna columna con inconsistencias de formato? Si sí, ¿cuál y qué código usarías para corregirla? Si no encontraste ninguna, dilo explícitamente — es una conclusión válida siempre que esté respaldada por lo que revisaste en 1.1.

_Tu respuesta:_ No encontré variantes causadas por diferencias entre mayúsculas, minúsculas o espacios. Los valores de cada categoría tienen una escritura consistente, por lo que no aplicaría una normalización con .str.strip() o .str.lower(). Los valores No internet service y No phone service no se modificaron porque no son errores de formato.

---
## Actividad 2 — Valores inválidos (20 pts)

Varias columnas de este dataset (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) tienen un tercer valor además de `'Yes'`/`'No'`: `'No internet service'`. De forma similar, `MultipleLines` tiene `'No phone service'`.

In [ ]:
df['OnlineSecurity'].value_counts()

,count
OnlineSecurity,
No,3498
Yes,2019
No internet service,1526


In [ ]:
# Cruzar OnlineSecurity con InternetService
comparacion = pd.crosstab(
    df['OnlineSecurity'],
    df['InternetService']
)

print(comparacion)

# Comprobar si ambos casos coinciden exactamente
sin_seguridad_por_falta_internet = (
    df['OnlineSecurity'] == 'No internet service'
)

sin_servicio_internet = (
    df['InternetService'] == 'No'
)

print("\n¿Las filas coinciden?",
      (sin_seguridad_por_falta_internet == sin_servicio_internet).all())

InternetService       DSL  Fiber optic    No
OnlineSecurity                              
No                   1241         2257     0
No internet service     0            0  1526
Yes                  1180          839     0

¿Las filas coinciden? True


**2.2 — Conclusión (responde aquí en Markdown):**

¿`'No internet service'` es un valor inválido (como `Absurd`/`YOLO` en la sesión de clase) o es una categoría legítima? Justifica tu respuesta con lo que verificaste en 2.1. ¿Tomarías alguna acción sobre esta columna, o la dejarías tal cual?

_Tu respuesta:_ No internet service es una categoría legítima y no un valor inválido. El cruce de las columnas mostró que los 1,526 registros con este valor en OnlineSecurity corresponden a clientes cuyo valor en InternetService es No. Esto indica que el cliente no puede tener seguridad en línea porque no cuenta con servicio de internet. Dejaría la categoría tal como está, ya que se puede distinguir entre un cliente que no contrató la seguridad y otro que no puede tenerla por falta de servicio.

---
## Actividad 3 — Alta cardinalidad (25 pts)

In [ ]:
# Obtener número total de filas
total_filas = len(df)

# Calcular los valores únicos de todas las columnas
valores_unicos = df.nunique()

# Crear una tabla con la cardinalidad y su razón
cardinalidad = pd.DataFrame({
    'Valores_unicos': valores_unicos,
    'Razon': valores_unicos / total_filas
})

# Ordenar desde la cardinalidad más alta
cardinalidad = cardinalidad.sort_values(
    by='Razon',
    ascending=False
)

print(cardinalidad)

                  Valores_unicos     Razon
customerID                  7043  1.000000
TotalCharges                6531  0.927304
MonthlyCharges              1585  0.225046
tenure                        73  0.010365
PaymentMethod                  4  0.000568
StreamingMovies                3  0.000426
TechSupport                    3  0.000426
OnlineBackup                   3  0.000426
StreamingTV                    3  0.000426
DeviceProtection               3  0.000426
MultipleLines                  3  0.000426
InternetService                3  0.000426
OnlineSecurity                 3  0.000426
Contract                       3  0.000426
Partner                        2  0.000284
SeniorCitizen                  2  0.000284
gender                         2  0.000284
Dependents                     2  0.000284
PhoneService                   2  0.000284
PaperlessBilling               2  0.000284
Churn                          2  0.000284


**3.2 — Conclusión (responde aquí en Markdown):**

¿Qué columna(s) tienen alta cardinalidad? Para la columna con mayor cardinalidad: ¿por qué nunca deberías usarla como variable predictora en un modelo, incluso si la codificaras? (relaciona tu respuesta con lo discutido en clase sobre identificadores únicos)

_Tu respuesta:_ La columna con mayor cardinalidad es customerID, ya que tiene 7,043 valores únicos en 7,043 filas, por lo que su razón es 1. También aparecen con cardinalidad alta TotalCharges y MonthlyCharges, pero en ellas es normal porque representan cantidades numéricas. customerID no debería utilizarse como variable predictora porque solamente identifica a cada cliente y no representa una característica que se repita. Aunque se codificara, el modelo podría memorizar clientes específicos en lugar de aprender patrones que puedan aplicarse a clientes nuevos.

**3.3 — Agrupación "Top 10 + Otros"**

En clase agrupaste `country` de Netflix Titles en sus 10 categorías más frecuentes + `'Otros'`, reduciendo su cardinalidad. Aplica la misma técnica aquí sobre la columna de mayor cardinalidad que identificaste en 3.1 (pista: `.value_counts().head(10)`, luego `.where()` + `.isin()`, igual que en el notebook de clase).

In [ ]:
# Obtener los 10 identificadores más frecuentes
top10_clientes = df['customerID'].value_counts().head(10)
categorias_top10 = top10_clientes.index.tolist()

# Conservar Top 10 y agrupar los demás como Otros
df['customerID_agrupado'] = df['customerID'].where(
    df['customerID'].isin(categorias_top10),
    'Otros'
)

# Comparar la cardinalidad antes y después
print("Cardinalidad original:",
      df['customerID'].nunique())

print("Cardinalidad después de agrupar:",
      df['customerID_agrupado'].nunique())

print("\nConteo de la nueva columna:")
print(df['customerID_agrupado'].value_counts())

Cardinalidad original: 7043
Cardinalidad después de agrupar: 11

Conteo de la nueva columna:
customerID_agrupado
Otros         7033
7590-VHVEG       1
5575-GNVDE       1
9837-FWLCH       1
1699-HPSBG       1
7203-OYKCT       1
1035-IPQPU       1
7398-LXGYX       1
2823-LKABH       1
8775-CEBBJ       1
3186-AJIEK       1
Name: count, dtype: int64


**3.4 — Conclusión (responde aquí en Markdown):**

Después de agrupar, ¿la columna resultante te parece útil para un modelo? Compara este caso con el de `country` en el notebook de clase: ¿por qué agrupar en "Top 10 + Otros" funciona bien para una variable como `country`, pero no resuelve el problema real de la columna que agrupaste aquí?

_Tu respuesta:_ Aunque la cardinalidad se redujo de 7,043 a 11 valores, customerID_agrupado no sería útil para un modelo. Todos los identificadores originales aparecen una sola vez, por lo que los diez valores conservados fueron seleccionados sin una frecuencia realmente mayor. En una variable como country, las categorías representan grupos reales que se repiten y conservar los países más frecuentes mantiene información útil. customerID solo identifica personas distintas y agruparlas como Otros no crea una característica con significado. Por esta razón, eliminaría customerID antes de construir el modelo.

---
## Actividad 4 — Tipos de dato (30 pts)

In [ ]:
df['TotalCharges'].dtype

dtype('O')

**4.1 — Investiga (responde en Markdown):**

`TotalCharges` contiene valores numéricos (montos en dólares), pero pandas la cargó como `object`, no como `float`. Investiga por qué — revisa si hay algún valor que no se vea como un número normal.

_Tu respuesta:_ La columna TotalCharges fue cargada como object porque contiene 11 registros con un espacio en blanco en lugar de un número. Estos registros corresponden a clientes con tenure igual a 0, es decir, clientes que todavía no han acumulado cargos. Por esta razón, pandas no pudo interpretar toda la columna como numérica y consideraría que esos espacios representan un total acumulado de 0.

In [ ]:
# Identificar registros con espacios en blanco
filas_vacias = df['TotalCharges'].astype(str).str.strip() == ''

print("Registros con TotalCharges vacío:",
      filas_vacias.sum())

# Mostrar registros problemáticos
display(
    df.loc[
        filas_vacias,
        ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']
    ]
)

# Convertir TotalCharges a numérico
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

# Los valores vacíos pertenecen a clientes con tenure igual a 0,
# por eso se reemplazan por 0 sin eliminar las filas
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Verificar la corrección
print("\nTipo de dato corregido:",
      df['TotalCharges'].dtype)

print("Valores nulos restantes:",
      df['TotalCharges'].isnull().sum())

Registros con TotalCharges vacío: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,



Tipo de dato corregido: float64
Valores nulos restantes: 0


In [ ]:
# Guardar columnas con 10 valores únicos o menos
columnas_category = []

for columna in df.columns:
    if df[columna].nunique() <= 10:
        columnas_category.append(columna)

print("Columnas de baja cardinalidad:")
print(columnas_category)

# Convertir columnas seleccionadas a category
for columna in columnas_category:
    df[columna] = df[columna].astype('category')

# Verificar los tipos de datos
print("\nTipos de datos después de la conversión:")
print(df[columnas_category].dtypes)

Columnas de baja cardinalidad:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']

Tipos de datos después de la conversión:
gender              category
SeniorCitizen       category
Partner             category
Dependents          category
PhoneService        category
MultipleLines       category
InternetService     category
OnlineSecurity      category
OnlineBackup        category
DeviceProtection    category
TechSupport         category
StreamingTV         category
StreamingMovies     category
Contract            category
PaperlessBilling    category
PaymentMethod       category
Churn               category
dtype: object


---
## Reflexión final (10 pts)

Con base en las 4 actividades anteriores, responde:

1. De las alertas que detectaste (formato, valores inválidos, cardinalidad, tipos), ¿cuál te pareció más fácil de decidir y cuál más difícil? ¿Por qué?
2. Si tuvieras que entregar este dataset ya "perfilado" a un compañero para que construya un modelo predictivo, ¿qué le dirías sobre `customerID` y sobre `TotalCharges`?

_Tu respuesta:_
1. La alerta más fácil de decidir fue la de formato, porque al revisar los valores de todas las columnas categóricas estaban escritos de manera consistente, sin diferencias de mayúsculas ni espacios adicionales. La más difícil fue decidir si No internet service era un valor inválido, ya que al principio parecía una categoría extra, pero al cruzar OnlineSecurity con InternetService, aparece únicamente en clientes que no tienen servicio de internet, por lo que es una categoría legítima y no debe eliminarse.

2. Le diria que no utilice customerID como variable predictora, porque tiene un valor diferente para cada cliente y solo funciona como identificador; incluso agrupándola en “Top 10 + Otros” no aporta información útil al modelo. Para TotalCharges, explicaría que originalmente se cargó como object debido a 11 espacios en blanco correspondientes a clientes con tenure igual a 0. Esos valores se reemplazaron por 0 y la columna se convirtió a tipo numérico, por lo que ya puede utilizarse en el análisis y en el modelo.

---
## Rúbrica de evaluación

| Actividad | Puntos | Criterio |
|---|---|---|
| 1. Formato y valores inconsistentes | 15 | Revisó todas las columnas categóricas (no solo una); código comentado y conclusión (1.2) respaldada por lo que se observó, no solo afirmada |
| 2. Valores inválidos | 20 | Verificó la relación entre columnas antes de concluir; la conclusión (2.2) justifica con evidencia, no solo con intuición |
| 3. Alta cardinalidad | 25 | Calcula `.nunique()` para todas las columnas; aplica correctamente el agrupamiento Top 10 + Otros; la conclusión (3.4) explica por qué agrupar no resuelve el problema de un identificador único |
| 4. Tipos de dato | 30 | Identifica la causa raíz del `dtype` incorrecto (4.1); corrige `TotalCharges` sin perder información; conversión a `category` justificada por cardinalidad, no aplicada al azar |
| Reflexión final | 10 | Conecta las 4 actividades entre sí; no es una respuesta genérica o intercambiable con cualquier dataset |
| **Total** | **100** | |

**Nota sobre las conclusiones:** cada actividad tiene su propia pregunta de conclusión (1.2, 2.2, 3.4, 4.1) — esas respuestas se califican como parte de la actividad correspondiente, no solo la Reflexión final. Comenta tu código donde tomes una decisión (por ejemplo, por qué elegiste cierto umbral o cierta corrección) — el comentario también cuenta dentro del puntaje de cada actividad.